In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/listings_clean.csv')
print(df.shape)
print(df.dtypes)
df.head()


In [ ]:
print("Nulls:\n", df.isnull().sum())
print("\nRent stats:\n", df['rent_monthly'].describe())
print("\nBHK counts:\n", df['bhk_type'].value_counts())
print("\nFurnishing:\n", df['furnishing'].value_counts())
print("\nZone:\n", df['city_zone'].value_counts())


In [ ]:
df = df.dropna(subset=['rent_monthly', 'area_sqft', 'bhk_type', 'furnishing', 'city_zone'])
df = df[df['city_zone'] != 'Other']
df = df[df['area_sqft'] > 0]
print(f"Clean rows: {len(df)}")


In [ ]:
bhk_map = {'1 RK': 0, '1 BHK': 1, '2 BHK': 2, '3 BHK': 3, '4 BHK': 4, '4+ BHK': 5}
df['bhk_encoded'] = df['bhk_type'].map(bhk_map)
df = df[df['bhk_encoded'].notna()]
print(df['bhk_encoded'].value_counts())


In [ ]:
furnish_map = {'Unfurnished': 0, 'Semi-Furnished': 1, 'Furnished': 2}
df['furnishing_encoded'] = df['furnishing'].map(furnish_map)
df = df[df['furnishing_encoded'].notna()]
print(df['furnishing_encoded'].value_counts())


In [ ]:
df['log_sqft'] = np.log1p(df['area_sqft'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['area_sqft'], bins=40, color='steelblue')
axes[0].set_title('Raw sqft')
axes[1].hist(df['log_sqft'], bins=40, color='coral')
axes[1].set_title('Log sqft')
plt.tight_layout()
plt.show()


In [ ]:
locality_counts = df['locality'].value_counts().rename('locality_count')
df = df.join(locality_counts, on='locality')
print(df['locality_count'].describe())


In [ ]:
# Compute on full dataset for EDA only
# Proper leak-safe encoding happens inside CV in train.py
zone_mean = df.groupby('city_zone')['rent_monthly'].mean()
df['zone_target_enc'] = df['city_zone'].map(zone_mean)
print(zone_mean.sort_values(ascending=False))


In [ ]:
df['zone_x_furnishing'] = df['zone_target_enc'] * df['furnishing_encoded']
print(df['zone_x_furnishing'].describe())


In [ ]:
FEATURES = [
    'bhk_encoded',
    'furnishing_encoded',
    'log_sqft',
    'locality_count',
    'zone_target_enc',
    'zone_x_furnishing',
]
TARGET = 'rent_monthly'

# Drop price_per_sqft — DATA LEAKAGE (rent ÷ sqft = target-derived)
X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f"Feature matrix: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nFeature correlations with rent:")
print(X.corrwith(y).sort_values(ascending=False))
X.head()


In [ ]:
df_model = df[FEATURES + [TARGET, 'locality', 'city_zone']].copy()
df_model.to_csv('../data/features.csv', index=False)
print("Saved → data/features.csv")
